# 12 — Qwen Vision: Visual Understanding

## Why Qwen Vision in addition to CLIP?

| Aspect | CLIP | Qwen Vision |
|---|---|---|
| Output | 512-dim embedding vector | Natural language description / structured JSON |
| Strength | Fast similarity matching across 4,681 products | Explicit attribute extraction (color, material, style) |
| Weakness | Black-box — cannot explain *why* images are similar | Slower, requires LLM inference per image |
| Use in pipeline | Retrieval (FAISS) | Visual understanding for re-ranking / query expansion |

CLIP is used for **retrieval**. Qwen Vision is used for **understanding** — extracting explicit, human-readable visual attributes that can be matched against user filters or used to improve re-ranking.

## Architecture position

```
User Image
  ├─ CLIP Image Encoder  → Image FAISS  → Candidate Pool  (retrieval)
  └─ Qwen Vision         → Visual Attributes              (understanding)
                                    ↓
                           Combined with Qwen LLM text understanding
                                    ↓
                           Score Fusion + Re-ranking
```

## Limitations
- Qwen Vision is run per-image at inference time — not precomputed for all 4,681 products.
- Visual attribute extraction depends on image quality and product clarity.
- The model cannot identify brand names from logos reliably at 2B scale.
- RTX 2050 (4 GB VRAM) can run 2B model in fp16 but only one image at a time.

## 1. Imports

In [1]:
import json
import re
import numpy as np
import pandas as pd
import faiss
import torch
from pathlib import Path
from PIL import Image
from transformers import (
    Qwen2VLForConditionalGeneration,
    AutoProcessor,
    CLIPModel,
    CLIPProcessor,
    CLIPTokenizer,
)
from qwen_vl_utils import process_vision_info

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch  : {torch.__version__}")
print(f"device : {DEVICE}")
if DEVICE == "cuda":
    p = torch.cuda.get_device_properties(0)
    print(f"GPU    : {p.name}")
    print(f"VRAM   : {p.total_memory/1024**3:.1f} GB")

torch  : 2.6.0+cu124
device : cuda
GPU    : NVIDIA GeForce RTX 2050
VRAM   : 4.0 GB


## 2. Paths

In [2]:
NOTEBOOK_DIR = Path(".").resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
PROCESSED    = PROJECT_ROOT / "data" / "processed"
FAISS_DIR    = PROCESSED / "faiss"

PRODUCTS_CSV       = PROCESSED / "products_ml_ready.csv"
IMAGE_FAISS_PATH   = FAISS_DIR  / "image_index.faiss"
IMAGE_MAPPING_CSV  = FAISS_DIR  / "image_index_mapping.csv"
TEXT_FAISS_PATH    = FAISS_DIR  / "text_index.faiss"
TEXT_MAPPING_CSV   = FAISS_DIR  / "text_index_mapping.csv"

for p in [PRODUCTS_CSV, IMAGE_FAISS_PATH, IMAGE_MAPPING_CSV,
          TEXT_FAISS_PATH, TEXT_MAPPING_CSV]:
    assert p.exists(), f"Missing: {p}"
    print(f"  OK  {p.relative_to(PROJECT_ROOT)}")

  OK  data\processed\products_ml_ready.csv
  OK  data\processed\faiss\image_index.faiss
  OK  data\processed\faiss\image_index_mapping.csv
  OK  data\processed\faiss\text_index.faiss
  OK  data\processed\faiss\text_index_mapping.csv


## 3. Load Data and FAISS Indexes

In [3]:
image_index = faiss.read_index(str(IMAGE_FAISS_PATH))
text_index  = faiss.read_index(str(TEXT_FAISS_PATH))

image_mapping_df = pd.read_csv(IMAGE_MAPPING_CSV)
text_mapping_df  = pd.read_csv(TEXT_MAPPING_CSV)

image_faiss_to_pid = dict(zip(image_mapping_df["faiss_index"], image_mapping_df["pid"]))
text_faiss_to_pid  = dict(zip(text_mapping_df["faiss_index"],  text_mapping_df["pid"]))

products_df     = pd.read_csv(PRODUCTS_CSV)
products_by_pid = products_df.set_index("pid")

CATEGORIES = sorted(products_df["main_category"].unique().tolist())
N_PRODUCTS  = len(products_df)

assert image_index.d == 512 and text_index.d == 512
assert image_index.ntotal == N_PRODUCTS

print(f"Products      : {N_PRODUCTS}")
print(f"Image FAISS   : dim={image_index.d}, ntotal={image_index.ntotal}")
print(f"Text FAISS    : dim={text_index.d}, ntotal={text_index.ntotal}")

Products      : 4681
Image FAISS   : dim=512, ntotal=4681
Text FAISS    : dim=512, ntotal=4681


## 4. Load Qwen2-VL Model

Using `Qwen/Qwen2-VL-2B-Instruct` in **float16 on CUDA**.
This model takes image + text input and produces natural-language output.

In [4]:
QWEN_VL_MODEL = "Qwen/Qwen2-VL-2B-Instruct"

print(f"Loading {QWEN_VL_MODEL} ...")
qwen_vl_processor = AutoProcessor.from_pretrained(QWEN_VL_MODEL)
qwen_vl_model = Qwen2VLForConditionalGeneration.from_pretrained(
    QWEN_VL_MODEL,
    torch_dtype=torch.float16,
    device_map="cuda",
)
qwen_vl_model.eval()

vram_used = torch.cuda.memory_allocated() / 1024**3
print(f"Qwen2-VL loaded on GPU")
print(f"VRAM used : {vram_used:.2f} GB")

Loading Qwen/Qwen2-VL-2B-Instruct ...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Qwen2-VL loaded on GPU
VRAM used : 4.12 GB


## 5. Visual Analysis Prompt

The prompt requests structured JSON with a fixed schema.  
It instructs the model to only describe what is **visually observable** — no inference of brand/price.

In [5]:
VISION_PROMPT = """Analyze this product image and return ONLY a valid JSON object.
Do not include any text outside the JSON. No markdown, no code fences.

Describe only what is visually observable. Do not infer or invent attributes.

JSON schema:
{
  "product_type": string or null,
  "colors": list of strings,
  "style": string or null,
  "pattern": string or null,
  "material_appearance": string or null,
  "gender_appearance": "men" | "women" | "unisex" | "children" | null,
  "key_visual_attributes": list of strings,
  "visual_description": string
}

Rules:
- colors: list all clearly visible colors
- style: e.g. casual, formal, sports, traditional, modern
- pattern: e.g. solid, striped, floral, printed, plain
- material_appearance: e.g. fabric, leather, metal, plastic, wooden
- gender_appearance: based only on visual design cues
- key_visual_attributes: 3-6 concise descriptors useful for search
- visual_description: one sentence summarizing what the product looks like
- Use null for fields you cannot determine visually"""

VISION_SCHEMA_DEFAULTS = {
    "product_type"         : None,
    "colors"               : [],
    "style"                : None,
    "pattern"              : None,
    "material_appearance"  : None,
    "gender_appearance"    : None,
    "key_visual_attributes": [],
    "visual_description"   : "",
}

print("Vision prompt and schema defined.")

Vision prompt and schema defined.


## 6. `analyze_image` — Visual Understanding Function

In [6]:
def _extract_json_from_output(raw: str) -> dict:
    """Extract first JSON object from model output, stripping markdown fences."""
    raw = re.sub(r"```(?:json)?\s*", "", raw).strip()
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON object in output: {raw[:200]}")
    return json.loads(match.group())


def _validate_vision_output(parsed: dict) -> dict:
    """Fill missing fields with defaults and coerce types."""
    result = {}
    for key, default in VISION_SCHEMA_DEFAULTS.items():
        val = parsed.get(key, default)
        # Ensure list fields are lists
        if isinstance(default, list) and not isinstance(val, list):
            val = [str(val)] if val else []
        # Validate gender_appearance
        if key == "gender_appearance" and val not in ("men","women","unisex","children",None):
            val = None
        result[key] = val
    return result


def analyze_image(image_path: str,
                  max_new_tokens: int = 350,
                  min_pixels: int = 256*28*28,
                  max_pixels: int = 512*28*28) -> dict:
    """
    Analyze a product image using Qwen2-VL and return structured visual attributes.

    Parameters
    ----------
    image_path    : str  — absolute or notebook-relative path to a product image
    max_new_tokens: int  — maximum tokens to generate
    min/max_pixels: int  — image resolution bounds for Qwen2-VL processor

    Returns
    -------
    dict with keys:
        image_path, product_type, colors, style, pattern,
        material_appearance, gender_appearance,
        key_visual_attributes, visual_description
    """
    # Resolve path
    path = Path(image_path)
    if not path.is_absolute():
        path = (NOTEBOOK_DIR / path).resolve()
    if not path.exists():
        raise FileNotFoundError(f"Image not found: {path}")

    # Build message
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": str(path),
             "min_pixels": min_pixels, "max_pixels": max_pixels},
            {"type": "text",  "text": VISION_PROMPT},
        ]
    }]

    text = qwen_vl_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = qwen_vl_processor(
        text=[text], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt"
    ).to(DEVICE)

    with torch.no_grad():
        output_ids = qwen_vl_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    generated = output_ids[0][inputs.input_ids.shape[1]:]
    raw_output = qwen_vl_processor.decode(generated, skip_special_tokens=True)

    try:
        parsed = _extract_json_from_output(raw_output)
    except (json.JSONDecodeError, ValueError) as e:
        print(f"  WARN: JSON parse failed for {path.name}: {e}")
        print(f"  Raw: {raw_output[:300]}")
        parsed = {}

    result = _validate_vision_output(parsed)
    result["image_path"] = str(image_path)
    return result


print("analyze_image defined.")

analyze_image defined.


## 7. Load CLIP (for Image Retrieval)

CLIP is loaded separately for retrieval. It shares GPU space with Qwen2-VL.
We run CLIP after Qwen Vision to avoid OOM — they are not used simultaneously.

In [7]:
CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"
print(f"Loading {CLIP_MODEL_NAME} ...")
clip_model     = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(DEVICE)
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)
clip_model.eval()
print(f"CLIP loaded. VRAM used: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

Loading openai/clip-vit-base-patch32 ...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIP loaded. VRAM used: 4.68 GB


## 8. CLIP Image Retrieval Functions

In [8]:
def _l2_normalize(vec: np.ndarray) -> np.ndarray:
    norm = np.linalg.norm(vec, axis=1, keepdims=True)
    return vec / np.clip(norm, 1e-10, None)


def encode_image_clip(image_path: str) -> np.ndarray:
    """Encode an image into a 512-dim L2-normalized CLIP embedding."""
    path = Path(image_path)
    if not path.is_absolute():
        path = (NOTEBOOK_DIR / path).resolve()
    img = Image.open(path).convert("RGB")
    inputs = clip_processor(images=[img], return_tensors="pt")
    pixel_values = inputs["pixel_values"].to(DEVICE)
    with torch.no_grad():
        vis = clip_model.vision_model(pixel_values=pixel_values)
        emb = clip_model.visual_projection(vis.pooler_output)
    return _l2_normalize(emb.cpu().float().numpy())


def encode_text_clip(query: str) -> np.ndarray:
    """Encode a text string into a 512-dim L2-normalized CLIP embedding."""
    from transformers import CLIPTokenizer as _Tok
    tok = _Tok.from_pretrained(CLIP_MODEL_NAME)
    tokens = tok([query], return_tensors="pt", padding=True, truncation=True, max_length=77)
    tokens = {k: v.to(DEVICE) for k, v in tokens.items()}
    with torch.no_grad():
        out = clip_model.text_model(**tokens)
        emb = clip_model.text_projection(out.pooler_output)
    return _l2_normalize(emb.cpu().float().numpy())


def faiss_image_search(query_vec: np.ndarray, top_k: int = 10) -> list[dict]:
    """Search the image FAISS index, return list of {pid, score}."""
    scores, indices = image_index.search(query_vec.astype(np.float32), top_k)
    results = []
    for fidx, score in zip(indices[0], scores[0]):
        if fidx == -1: continue
        pid = image_faiss_to_pid.get(int(fidx))
        if pid and pid in products_by_pid.index:
            results.append({"pid": pid, "score": float(score)})
    return results


def faiss_text_search(query_vec: np.ndarray, top_k: int = 10) -> list[dict]:
    """Search the text FAISS index, return list of {pid, score}."""
    scores, indices = text_index.search(query_vec.astype(np.float32), top_k)
    results = []
    for fidx, score in zip(indices[0], scores[0]):
        if fidx == -1: continue
        pid = text_faiss_to_pid.get(int(fidx))
        if pid and pid in products_by_pid.index:
            results.append({"pid": pid, "score": float(score)})
    return results


def _attach_metadata(raw_results: list[dict], score_col: str = "score") -> pd.DataFrame:
    rows = []
    for rank, r in enumerate(raw_results, 1):
        meta = products_by_pid.loc[r["pid"]]
        rows.append({
            "rank"          : rank,
            "pid"           : r["pid"],
            "product_name"  : meta["product_name"],
            "main_category" : meta["main_category"],
            "brand"         : meta.get("brand", "Unknown"),
            "image_path"    : meta["image_path"],
            score_col       : round(r["score"], 4),
        })
    return pd.DataFrame(rows)


print("CLIP retrieval functions loaded.")

CLIP retrieval functions loaded.


## 9. Select Test Images Programmatically

One representative image per category sampled dynamically — no hardcoded PIDs.

In [9]:
rng = np.random.default_rng(42)

# Pick one product per category
test_images = []
for category in CATEGORIES:
    cat_products = products_df[products_df["main_category"] == category]
    sample = cat_products.sample(1, random_state=42).iloc[0]
    test_images.append({
        "pid"          : sample["pid"],
        "product_name" : sample["product_name"],
        "category"     : category,
        "image_path"   : sample["image_path"],
    })

print(f"Test images selected: {len(test_images)} (one per category)")
for t in test_images:
    print(f"  [{t['category']:35s}] {t['pid']}  {t['product_name'][:40]}")

Test images selected: 13 (one per category)
  [Automotive                         ] CRTECN2QYFYRSXTH  Allure Auto CM 1995 Car Mat Chevrolet Sa
  [Baby Care                          ] BLAE8BUKXGJ3DHMG  Offspring Printed Single Wrapper Multico
  [Beauty and Personal Care           ] CAGE8SCPX394WVJT  Nike Brown Combo Set
  [Clothing                           ] TSHE4S26FVNGW5TZ  CampusMall Printed Men's Round Neck T-Sh
  [Computers                          ] RTRDP2GSBX4AF7FK  TP-LINK 300 Mbps Universal WiFi
  [Footwear                           ] SNDEJTNXZ6YGMZMZ  SHOPOJ Women Bellies
  [Home Decor & Festive Needs         ] ARPEEMCAMGADRCGQ  Decore Sources Assorted Artificial Plant
  [Home Furnishing                    ] BLAED242BBPBXMC7  Rays007 Cartoon Double Quilts & Comforte
  [Jewellery                          ] NKCEJPENXNMT9KKS  JK Import Rudrash Alloy Mala Alloy, Wood
  [Kitchen & Dining                   ] PACDZUDMCHKDHGXA  Tescoma Presto Wheel Pizza Cutter
  [Mobiles & Accessori

## 10. Run Qwen Vision on All Test Images

In [10]:
vision_results = []

for item in test_images:
    print(f"Analyzing: [{item['category']}] {item['product_name'][:45]}")
    analysis = analyze_image(item["image_path"])
    analysis["pid"]          = item["pid"]
    analysis["product_name"] = item["product_name"]
    analysis["category"]     = item["category"]
    vision_results.append(analysis)

    print(f"  product_type  : {analysis['product_type']}")
    print(f"  colors        : {analysis['colors']}")
    print(f"  style         : {analysis['style']}")
    print(f"  pattern       : {analysis['pattern']}")
    print(f"  material      : {analysis['material_appearance']}")
    print(f"  gender_appear : {analysis['gender_appearance']}")
    print(f"  key_attrs     : {analysis['key_visual_attributes']}")
    print(f"  description   : {analysis['visual_description']}")
    print()

print(f"Vision analysis complete: {len(vision_results)} images.")

Analyzing: [Automotive] Allure Auto CM 1995 Car Mat Chevrolet Sail


  product_type  : None
  colors        : ['brown']
  style         : unisex
  pattern       : plain
  material      : rubber
  gender_appear : unisex
  key_attrs     : ['rubber mat with circular dots', 'brown color']
  description   : The product is a set of four brown rubber car mats with circular dots, designed for unisex use.

Analyzing: [Baby Care] Offspring Printed Single Wrapper Multicolor


  product_type  : None
  colors        : ['white']
  style         : unisex
  pattern       : bear
  material      : fabric
  gender_appear : children
  key_attrs     : ['bear pattern', 'white background']
  description   : A white blanket with a bear pattern and a yellow border.

Analyzing: [Beauty and Personal Care] Nike Brown Combo Set


  product_type  : deodorant
  colors        : ['brown']
  style         : unisex
  pattern       : solid
  material      : plastic
  gender_appear : unisex
  key_attrs     : ['brown bottle with blue and orange logo']
  description   : A brown bottle with a blue and orange Nike logo on it, designed for men.

Analyzing: [Clothing] CampusMall Printed Men's Round Neck T-Shirt


  product_type  : None
  colors        : ['white']
  style         : unisex
  pattern       : plain
  material      : fabric
  gender_appear : unisex
  key_attrs     : ["Keep Calm and Don't Care text on a white t-shirt with an orange background"]
  description   : A white t-shirt with the text 'Keep Calm and Don't Care' on an orange background, featuring a crown design in the center.

Analyzing: [Computers] TP-LINK 300 Mbps Universal WiFi


...

  product_type  : None
  colors        : ['gold']
  style         : traditional
  pattern       : plain
  material      : metal
  gender_appear : unisex
  key_attrs     : ['gold chain with rudraksha beads']
  description   : A gold chain with rudraksha beads, a traditional design, and a unisex appearance.

Analyzing: [Kitchen & Dining] Tescoma Presto Wheel Pizza Cutter


  product_type  : None
  colors        : ['white']
  style         : modern
  pattern       : solid
  material      : metal
  gender_appear : unisex
  key_attrs     : ['pizza cutter', 'white handle', 'metal blade']
  description   : A modern, white pizza cutter with a metal blade and a handle for easy grip.

Analyzing: [Mobiles & Accessories] Enthopia Flip Cover for iPad 2, iPad 3, iPad 


  product_type  : tablet_case
  colors        : ['brown', 'green', 'red', 'purple', 'white']
  style         : modern
  pattern       : solid
  material      : plastic
  gender_appear : unisex
  key_attrs     : ['hippie', 'smoking', 'peace sign']
  description   : A modern, plastic tablet case with a hippie character design, featuring a brown and green color scheme, a peace sign, and a smoking character.

Analyzing: [Tools & Hardware] Skys&Ray Plastic Toothbrush Holder


  product_type  : None
  colors        : ['white']
  style         : modern
  pattern       : solid
  material      : plastic
  gender_appear : unisex
  key_attrs     : ['soap holder', 'suction cup']
  description   : A modern, white soap holder with a blue cover and a suction cup for easy attachment to the wall.

Analyzing: [Watches] Jazma E11A835LA Casual Analog Watch  - For Me


  product_type  : None
  colors        : ['white']
  style         : traditional
  pattern       : solid
  material      : metal
  gender_appear : unisex
  key_attrs     : ['white face with black roman numerals', 'black leather strap']
  description   : A traditional, white-faced watch with black leather straps, featuring a silver metal case and a black leather strap with a silver metal buckle.

Vision analysis complete: 13 images.


## 11. Visual Analysis Summary Table

In [11]:
pd.set_option("display.max_colwidth", 50)

summary_rows = []
for r in vision_results:
    summary_rows.append({
        "category"     : r["category"],
        "product"      : r["product_name"][:40],
        "product_type" : r["product_type"] or "-",
        "colors"       : ", ".join(r["colors"]) if r["colors"] else "-",
        "style"        : r["style"] or "-",
        "material"     : r["material_appearance"] or "-",
        "visual_desc"  : r["visual_description"][:55] if r["visual_description"] else "-",
    })

summary_df = pd.DataFrame(summary_rows)
print("Qwen Vision Analysis Summary:")
print(summary_df.to_string(index=False))

Qwen Vision Analysis Summary:
                  category                                  product            product_type                           colors       style material                                             visual_desc
                Automotive Allure Auto CM 1995 Car Mat Chevrolet Sa                       -                            brown      unisex   rubber The product is a set of four brown rubber car mats with
                 Baby Care Offspring Printed Single Wrapper Multico                       -                            white      unisex   fabric A white blanket with a bear pattern and a yellow border
  Beauty and Personal Care                     Nike Brown Combo Set               deodorant                            brown      unisex  plastic A brown bottle with a blue and orange Nike logo on it, 
                  Clothing CampusMall Printed Men's Round Neck T-Sh                       -                            white      unisex   fabric A white t-shirt 

## 12. Integrate with Retrieval Pipeline

For each test image, run the full pipeline:
1. Qwen Vision → visual attributes
2. CLIP Image Encoder → 512-dim embedding
3. FAISS Image Index → top-K visually similar products
4. Also run CLIP text search using Qwen's `visual_description` as query

This shows both retrieval signals side by side.

In [12]:
def vision_retrieve(image_path: str, top_k: int = 5) -> dict:
    """
    Full vision retrieval pipeline:
    1. Qwen Vision → structured attributes
    2. CLIP image embed → FAISS image search
    3. CLIP text embed (from visual_description) → FAISS text search

    Returns
    -------
    dict with keys:
        vision_analysis   : structured Qwen output
        image_candidates  : DataFrame from image FAISS
        text_candidates   : DataFrame from text FAISS (using visual_description)
    """
    # Step 1 — Qwen Vision analysis
    vision_analysis = analyze_image(image_path)

    # Step 2 — CLIP image embedding → image FAISS
    img_vec = encode_image_clip(image_path)
    img_raw = faiss_image_search(img_vec, top_k=top_k)
    image_candidates = _attach_metadata(img_raw, score_col="image_score")

    # Step 3 — CLIP text embedding from visual_description → text FAISS
    visual_desc = vision_analysis.get("visual_description", "")
    if not visual_desc:
        # Fallback: build query from key attributes
        parts = ([vision_analysis.get("product_type")] +
                 vision_analysis.get("colors", []) +
                 vision_analysis.get("key_visual_attributes", []))
        visual_desc = " ".join(p for p in parts if p)

    text_vec = encode_text_clip(visual_desc) if visual_desc else None
    if text_vec is not None:
        txt_raw = faiss_text_search(text_vec, top_k=top_k)
        text_candidates = _attach_metadata(txt_raw, score_col="text_score")
    else:
        text_candidates = pd.DataFrame()

    return {
        "vision_analysis"  : vision_analysis,
        "image_candidates" : image_candidates,
        "text_candidates"  : text_candidates,
    }


print("vision_retrieve defined.")

vision_retrieve defined.


## 13. Run Vision Retrieval on Sample Products

In [13]:
# Run on first 4 test images (one per diverse category)
diverse_samples = test_images[:4]
retrieval_results = []

for item in diverse_samples:
    print("=" * 65)
    print(f"Query image : {item['image_path']}")
    print(f"Product     : {item['product_name']} [{item['category']}]")
    print()

    result = vision_retrieve(item["image_path"], top_k=5)
    va = result["vision_analysis"]

    print(f"Qwen Vision:")
    print(f"  product_type : {va['product_type']}")
    print(f"  colors       : {va['colors']}")
    print(f"  description  : {va['visual_description']}")
    print()

    print(f"Image FAISS top-5 (CLIP image similarity):")
    ic = result["image_candidates"]
    if not ic.empty:
        # Exclude query product itself from display
        ic_filtered = ic[ic["pid"] != item["pid"]].head(5)
        print(ic_filtered[["rank","product_name","main_category","image_score"]].to_string(index=False))
    print()

    print(f"Text FAISS top-5 (CLIP text, query=visual_description):")
    tc = result["text_candidates"]
    if not tc.empty:
        print(tc[["rank","product_name","main_category","text_score"]].to_string(index=False))
    print()

    retrieval_results.append(result)

print("Retrieval integration complete.")

Query image : ../data/images/CRTECN2QYFYRSXTH.jpg
Product     : Allure Auto CM 1995 Car Mat Chevrolet Sail [Automotive]



Qwen Vision:
  product_type : None
  colors       : ['brown']
  description  : The product is a set of four brown rubber car mats with circular dots, designed for unisex use.

Image FAISS top-5 (CLIP image similarity):
 rank                             product_name main_category  image_score
    1 Allure Auto CM 2083 Car Mat Nissan Micra    Automotive          1.0
    2    Allure Auto CM 2105 Car Mat Tata Nano    Automotive          1.0
    3  Allure Auto CM 2091 Car Mat Skoda Fabia    Automotive          1.0
    4 Allure Auto CM 2018 Car Mat Honda Accord    Automotive          1.0
    5  Allure Auto CM 2019 Car Mat Honda Amaze    Automotive          1.0

Text FAISS top-5 (CLIP text, query=visual_description):
 rank                                    product_name main_category  text_score
    1 3a Autocare Rubber Mat Car Mat Suzuki New Swift    Automotive      0.6259
    2      4D Mats Fresh Interior Car Mat Skoda Laura    Automotive      0.6002
    3                 3a AUTOCARE Car Ma

Qwen Vision:
  product_type : None
  colors       : ['white']
  description  : A white blanket with a bear pattern and a yellow border.

Image FAISS top-5 (CLIP image similarity):
 rank                                                                  product_name         main_category  image_score
    2 MeeMee 100% Cotton Small Sleeping Mat Mat Baby Mattress Set with Mosquito Net             Baby Care       0.8428
    3                                                       Gee & Bee Girl's Pyjama              Clothing       0.8084
    4                                       Welhouse Geometric Double Blanket Black       Home Furnishing       0.8049
    5         Fastway Book Cover for Micromax Canvas Tab P690 8GB 3G Calling Tablet Mobiles & Accessories       0.8019

Text FAISS top-5 (CLIP text, query=visual_description):
 rank                                  product_name   main_category  text_score
    1                     Trident Cotton Face Towel Home Furnishing      0.4948
    2   

Qwen Vision:
  product_type : deodorant
  colors       : ['brown']
  description  : A brown bottle with a blue and orange Nike logo on it, designed for men.

Image FAISS top-5 (CLIP image similarity):
 rank              product_name            main_category  image_score
    2     Nike Indigo Combo Set Beauty and Personal Care       0.8205
    3     Nike Azzure Combo Set Beauty and Personal Care       0.7697
    4 Nike Urban Musk Combo Set Beauty and Personal Care       0.7673
    5   Nike Original Combo Set Beauty and Personal Care       0.7584

Text FAISS top-5 (CLIP text, query=visual_description):
 rank                                           product_name    main_category  text_score
    1                                         Shoe Day Boots         Footwear      0.5329
    2      Apoxy APX-RUNNER-2-KIDS-BLUE-ORANGE Running Shoes         Footwear      0.5002
    3 Blue Birds Usa Homeware Stainless Steel 1000 ml Bottle Kitchen & Dining      0.4970
    4                           

Qwen Vision:
  product_type : None
  colors       : ['white']
  description  : A white t-shirt with the text 'Keep Calm and Don't Care' on an orange background, featuring a crown design in the center.

Image FAISS top-5 (CLIP image similarity):
 rank                                      product_name main_category  image_score
    2       CampusMall Printed Men's Round Neck T-Shirt      Clothing       0.8912
    3 Ocean Race Graphic Print Men's Round Neck T-Shirt      Clothing       0.7583
    4 Ocean Race Graphic Print Men's Round Neck T-Shirt      Clothing       0.7502
    5 Ocean Race Graphic Print Men's Round Neck T-Shirt      Clothing       0.7376

Text FAISS top-5 (CLIP text, query=visual_description):
 rank                                       product_name main_category  text_score
    1 Orange and Orchid Printed Men's Round Neck T-Shirt      Clothing      0.4848
    2 Orange and Orchid Printed Men's Round Neck T-Shirt      Clothing      0.4848
    3       Orange Plum Printed Me

## 14. Validation

In [14]:
required_keys = list(VISION_SCHEMA_DEFAULTS.keys()) + ["image_path"]

print("=== Validation ===")
for i, r in enumerate(vision_results):
    for key in required_keys:
        assert key in r, f"Result {i}: missing key '{key}'"
    assert isinstance(r["colors"], list),               f"Result {i}: colors must be list"
    assert isinstance(r["key_visual_attributes"], list), f"Result {i}: key_visual_attributes must be list"
    if r["gender_appearance"] is not None:
        assert r["gender_appearance"] in ("men","women","unisex","children"), \
            f"Result {i}: invalid gender_appearance"

print(f"  Schema validation: {len(vision_results)} results  ✓")

# Validate CUDA was actually used
assert torch.cuda.is_available(), "CUDA not available"
vram_after = torch.cuda.memory_allocated() / 1024**3
print(f"  CUDA active: True  ✓")
print(f"  VRAM used after run: {vram_after:.2f} GB")

# Validate retrieval results
for i, rr in enumerate(retrieval_results):
    ic = rr["image_candidates"]
    if not ic.empty:
        assert ic["pid"].duplicated().sum() == 0, f"Retrieval {i}: dup PIDs"
        bad = [p for p in ic["pid"] if p not in products_by_pid.index]
        assert len(bad) == 0, f"Retrieval {i}: invalid PIDs"

print(f"  Retrieval PID validation: {len(retrieval_results)} queries  ✓")
print("All validations passed.")

=== Validation ===
  Schema validation: 13 results  ✓
  CUDA active: True  ✓
  VRAM used after run: 4.69 GB
  Retrieval PID validation: 4 queries  ✓
All validations passed.


## 15. Final Report

In [15]:
n_with_product_type = sum(1 for r in vision_results if r["product_type"])
n_with_colors       = sum(1 for r in vision_results if r["colors"])
n_with_style        = sum(1 for r in vision_results if r["style"])
n_with_material     = sum(1 for r in vision_results if r["material_appearance"])
n_with_gender       = sum(1 for r in vision_results if r["gender_appearance"])
n_with_desc         = sum(1 for r in vision_results if r["visual_description"])
n = len(vision_results)

print("=" * 60)
print("QWEN VISION UNDERSTANDING — FINAL REPORT")
print("=" * 60)
print(f"Model               : {QWEN_VL_MODEL}")
print(f"Device              : {DEVICE} (RTX 2050)")
print(f"VRAM after run      : {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"Images analyzed     : {n}")
print()
print("Field extraction rate:")
print(f"  product_type         : {n_with_product_type}/{n}")
print(f"  colors               : {n_with_colors}/{n}")
print(f"  style                : {n_with_style}/{n}")
print(f"  material_appearance  : {n_with_material}/{n}")
print(f"  gender_appearance    : {n_with_gender}/{n}")
print(f"  visual_description   : {n_with_desc}/{n}")
print()
print("Functions implemented:")
print("  analyze_image(path)        → structured visual dict")
print("  encode_image_clip(path)    → 512-dim CLIP embedding")
print("  encode_text_clip(query)    → 512-dim CLIP embedding")
print("  faiss_image_search(vec, k) → FAISS image candidates")
print("  faiss_text_search(vec, k)  → FAISS text candidates")
print("  vision_retrieve(path, k)   → unified retrieval result")
print()
print("Pipeline flow:")
print("  Image → Qwen2-VL → visual_description")
print("  Image → CLIP vision → FAISS image search")
print("  visual_description → CLIP text → FAISS text search")
print()
print("Limitations:")
print("  - Per-image inference (not precomputed) adds latency")
print("  - 2B model may miss fine-grained product distinctions")
print("  - VRAM is near-limit: cannot run Qwen2-VL and Qwen2.5 simultaneously")
print("  - Brand logo recognition is unreliable at this scale")
print()
print("Files created/modified:")
print("  notebooks/12_qwen_vision_understanding.ipynb")
print()
print("Validation: PASSED")
print("Status: COMPLETE")
print("Next stage: Backend API")
print("=" * 60)

QWEN VISION UNDERSTANDING — FINAL REPORT
Model               : Qwen/Qwen2-VL-2B-Instruct
Device              : cuda (RTX 2050)
VRAM after run      : 4.69 GB
Images analyzed     : 13

Field extraction rate:
  product_type         : 3/13
  colors               : 12/13
  style                : 12/13
  material_appearance  : 12/13
  gender_appearance    : 12/13
  visual_description   : 12/13

Functions implemented:
  analyze_image(path)        → structured visual dict
  encode_image_clip(path)    → 512-dim CLIP embedding
  encode_text_clip(query)    → 512-dim CLIP embedding
  faiss_image_search(vec, k) → FAISS image candidates
  faiss_text_search(vec, k)  → FAISS text candidates
  vision_retrieve(path, k)   → unified retrieval result

Pipeline flow:
  Image → Qwen2-VL → visual_description
  Image → CLIP vision → FAISS image search
  visual_description → CLIP text → FAISS text search

Limitations:
  - Per-image inference (not precomputed) adds latency
  - 2B model may miss fine-grained prod